# Eye Detection and Motion Tracking

In [4]:
import cv2
import numpy as np
import os

In [ ]:
import cv2

# --------------------------------------------------
# CAMERA
# --------------------------------------------------

cap = cv2.VideoCapture(0)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1200)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
cap.set(cv2.CAP_PROP_FPS, 30)
h,w=720,1200
if not cap.isOpened():
    print("Error: Could not open camera")
    exit()

# --------------------------------------------------
# MAIN LOOP
# --------------------------------------------------

while True:

    ret, frame = cap.read()

    if not ret:
        print("Error: Could not read frame")
        break

    # Frame dimensions
    rows, cols =frame.shape[:2] 

    # --------------------------------------------------
    # ROI
    # --------------------------------------------------

    roi = frame[
        300:1000,
        100:650
    ]

    # --------------------------------------------------
    # GRAYSCALE
    # --------------------------------------------------

    gray = cv2.cvtColor(
        roi,
        cv2.COLOR_BGR2GRAY
    )

    # --------------------------------------------------
    # GAUSSIAN BLUR
    # --------------------------------------------------

    blur = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    # --------------------------------------------------
    # THRESHOLD
    # --------------------------------------------------

    _, thresh = cv2.threshold(
        blur,
        80,
        255,
        cv2.THRESH_BINARY_INV
    )

    # --------------------------------------------------
    # FIND CONTOURS
    # --------------------------------------------------

    contours, hierarchy = cv2.findContours(
        thresh,
        cv2.RETR_TREE,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # --------------------------------------------------
    # PROCESS CONTOURS
    # --------------------------------------------------

    for cnt in contours:

        # Bounding rectangle
        x, y, w, h = cv2.boundingRect(cnt)

        # Area
        area = cv2.contourArea(cnt)

        # --------------------------------------------------
        # FILTER
        # --------------------------------------------------

        if w > 10 and h > 10 and area > 100:

            # Center of detected object
            center_x = x + w // 2
            center_y = y + h // 2

            # --------------------------------------------------
            # RECTANGLE
            # --------------------------------------------------

            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (255, 0, 0),
                2
            )

            # --------------------------------------------------
            # CENTER CIRCLE
            # --------------------------------------------------

            cv2.circle(
                frame,
                (center_x, center_y),
                5,
                (0, 255, 0),
                -1
            )

            # --------------------------------------------------
            # HORIZONTAL LINE
            # --------------------------------------------------

            cv2.line(
                frame,
                (0, center_y),
                (w, center_y),
                (0, 0, 255),
                2
            )

            # --------------------------------------------------
            # VERTICAL LINE
            # --------------------------------------------------

            cv2.line(
                frame,
                (center_x, 0),
                (center_x, h),
                (0, 0, 255),
                2
            )

            # --------------------------------------------------
            # OBJECT INFORMATION
            # --------------------------------------------------

            cv2.putText(
                frame,
                f"X:{center_x} Y:{center_y}",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

            # Print information
            print(
                f"Area: {area:.0f} | "
                f"Object: ({center_x}, {center_y})",
                end="\r"
            )
 

    # --------------------------------------------------
    # DISPLAY
    # --------------------------------------------------

    cv2.imshow("Frame", frame)
    cv2.imshow("ROI", roi)
    cv2.imshow("Gray", gray)
    cv2.imshow("Threshold", thresh)

    # --------------------------------------------------
    # EXIT
    # --------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# --------------------------------------------------
# RELEASE
# --------------------------------------------------

cap.release()
cv2.destroyAllWindows()